# 03 PyQSAR3 Modeling

## Step 3.1: Strict Data Pre-Filtering

This notebook performs only the Phase 3.1 descriptor pre-filtering step. It does not run PyQSAR3 feature selection, clustering, model fitting, cross-validation, consensus modeling, or external validation.

Golden rule: every filter is fitted strictly on the Training set. The exact same feature columns are then retained or dropped from the Test set to prevent data leakage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TRAIN_IN = PROJECT_ROOT / "data" / "features" / "mordred_train.csv"
TEST_IN = PROJECT_ROOT / "data" / "features" / "mordred_test.csv"
TRAIN_OUT = PROJECT_ROOT / "data" / "features" / "filtered_train.csv"
TEST_OUT = PROJECT_ROOT / "data" / "features" / "filtered_test.csv"

ID_COLUMNS = ["SMILES", "logKoc"]
CORRELATION_THRESHOLD = 0.90

assert TRAIN_IN.exists(), f"Missing input file: {TRAIN_IN}"
assert TEST_IN.exists(), f"Missing input file: {TEST_IN}"


## Load Raw Mordred Features and Separate Metadata

The raw Mordred tables are loaded from Phase 2. `SMILES` and `logKoc` are preserved separately while descriptor filters are applied only to the feature matrix.

In [2]:
train_raw = pd.read_csv(TRAIN_IN)
test_raw = pd.read_csv(TEST_IN)

assert train_raw.shape == (514, 1615), f"Unexpected train input shape: {train_raw.shape}"
assert test_raw.shape == (128, 1615), f"Unexpected test input shape: {test_raw.shape}"
assert list(train_raw.columns) == list(test_raw.columns), "Train/Test input schemas differ"
assert train_raw.columns[:2].tolist() == ID_COLUMNS, f"Expected first columns {ID_COLUMNS}"

train_meta = train_raw[ID_COLUMNS].copy()
test_meta = test_raw[ID_COLUMNS].copy()
X_train = train_raw.drop(columns=ID_COLUMNS).copy()
X_test = test_raw.drop(columns=ID_COLUMNS).copy()

original_train_shape = train_raw.shape
original_test_shape = test_raw.shape
original_feature_count = X_train.shape[1]

print(f"Original Train shape: {original_train_shape}")
print(f"Original Test shape: {original_test_shape}")
print(f"Original descriptor feature count: {original_feature_count}")
display(X_train.head())


Original Train shape: (514, 1615)
Original Test shape: (128, 1615)
Original descriptor feature count: 1613


,ABC,ABCGG,nAcid,nBase,SpAbs_A,SpMax_A,SpDiam_A,SpAD_A,SpMAD_A,LogEE_A,...,SRW10,TSRW10,MW,AMW,WPath,WPol,Zagreb1,Zagreb2,mZagreb1,mZagreb2
0,2.121320,2.340100,0,0,4.472136,1.618034,3.236068,4.472136,1.118034,2.155909,...,5.509388,22.328143,56.026215,7.003277,10,1,10.0,8.0,2.500000,1.250000
1,2.449490,2.449490,1,0,3.464102,1.732051,3.464102,3.464102,0.866025,2.178059,...,6.188264,24.179697,60.021129,7.502641,9,0,12.0,9.0,3.111111,1.000000
2,9.570086,9.944926,0,0,16.030562,2.262124,4.524249,16.030562,1.145040,3.482048,...,9.082166,44.461407,223.060959,7.966463,332,17,60.0,64.0,7.534722,3.458333
3,1.414214,1.414214,0,0,2.828427,1.414214,2.828427,2.828427,0.942809,1.849457,...,4.174387,17.310771,46.041865,5.115763,4,0,6.0,4.0,2.250000,1.000000
4,3.047207,3.305183,0,0,5.226252,1.847759,3.695518,5.226252,1.045250,2.408576,...,6.834109,27.254130,76.052429,5.850187,18,2,16.0,14.0,3.361111,1.333333


## Step A: Missing-Value Filter Fitted on Training Set

Descriptor columns containing any `NaN` in the Training set are removed from both Training and Test sets. Test-set missingness is not used to decide which columns to drop.

In [3]:
nan_columns = X_train.columns[X_train.isna().any()].tolist()

X_train_nan = X_train.drop(columns=nan_columns)
X_test_nan = X_test.drop(columns=nan_columns)

print(f"NaN-filter removed features: {len(nan_columns)}")
print(f"Remaining features after NaN filter: {X_train_nan.shape[1]}")

assert X_train_nan.shape[1] == X_test_nan.shape[1], "Train/Test feature counts differ after NaN filter"
assert list(X_train_nan.columns) == list(X_test_nan.columns), "Train/Test columns differ after NaN filter"
assert not X_train_nan.isna().any().any(), "Training NaNs remain after NaN filter"


NaN-filter removed features: 491
Remaining features after NaN filter: 1122


## Step B: Zero-Variance Filter Fitted on Training Set

`VarianceThreshold(threshold=0.0)` is fitted only on the Training features remaining after the missing-value filter. The selected Training columns are then applied directly to the Test set.

In [4]:
variance_filter = VarianceThreshold(threshold=0.0)
variance_filter.fit(X_train_nan)

variance_keep_mask = variance_filter.get_support()
variance_keep_columns = X_train_nan.columns[variance_keep_mask].tolist()
zero_variance_columns = X_train_nan.columns[~variance_keep_mask].tolist()

X_train_var = pd.DataFrame(
    variance_filter.transform(X_train_nan),
    columns=variance_keep_columns,
    index=X_train_nan.index,
)
X_test_var = X_test_nan[variance_keep_columns].copy()

print(f"Zero-variance filter removed features: {len(zero_variance_columns)}")
print(f"Remaining features after zero-variance filter: {X_train_var.shape[1]}")

assert X_train_var.shape[1] == X_test_var.shape[1], "Train/Test feature counts differ after variance filter"
assert list(X_train_var.columns) == list(X_test_var.columns), "Train/Test columns differ after variance filter"


Zero-variance filter removed features: 165
Remaining features after zero-variance filter: 957


## Step C: High-Correlation Filter Fitted on Training Set

Pearson correlations are calculated only from the Training features remaining after the NaN and zero-variance filters. For each highly correlated pair with absolute correlation greater than 0.90, the later feature in column order is marked for removal. Those Training-selected columns are removed from both splits.

In [5]:
corr_matrix = X_train_var.corr(method="pearson").abs()
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
correlation_drop_columns = [
    column for column in upper_triangle.columns
    if (upper_triangle[column] > CORRELATION_THRESHOLD).any()
]

X_train_filtered = X_train_var.drop(columns=correlation_drop_columns)
X_test_filtered = X_test_var.drop(columns=correlation_drop_columns)

print(f"Correlation filter threshold: abs(r) > {CORRELATION_THRESHOLD}")
print(f"Correlation filter removed features: {len(correlation_drop_columns)}")
print(f"Remaining features after correlation filter: {X_train_filtered.shape[1]}")

assert X_train_filtered.shape[1] == X_test_filtered.shape[1], "Train/Test feature counts differ after correlation filter"
assert list(X_train_filtered.columns) == list(X_test_filtered.columns), "Train/Test columns differ after correlation filter"


Correlation filter threshold: abs(r) > 0.9
Correlation filter removed features: 578
Remaining features after correlation filter: 379


## Merge Metadata and Save Filtered Outputs

The preserved `SMILES` and `logKoc` columns are concatenated back to the filtered descriptor matrices. The resulting files are the only outputs of Step 3.1.

In [6]:
filtered_train = pd.concat(
    [train_meta.reset_index(drop=True), X_train_filtered.reset_index(drop=True)],
    axis=1,
)
filtered_test = pd.concat(
    [test_meta.reset_index(drop=True), X_test_filtered.reset_index(drop=True)],
    axis=1,
)

assert filtered_train.shape[0] == train_raw.shape[0], "Train row count changed"
assert filtered_test.shape[0] == test_raw.shape[0], "Test row count changed"
assert list(filtered_train.columns) == list(filtered_test.columns), "Filtered Train/Test schemas differ"
assert filtered_train.columns[:2].tolist() == ID_COLUMNS, "Metadata columns are not preserved at the front"

TRAIN_OUT.parent.mkdir(parents=True, exist_ok=True)
filtered_train.to_csv(TRAIN_OUT, index=False)
filtered_test.to_csv(TEST_OUT, index=False)

print(f"Saved filtered Train dataset: {TRAIN_OUT}")
print(f"Saved filtered Test dataset: {TEST_OUT}")
print(f"Final Train shape: {filtered_train.shape}")
print(f"Final Test shape: {filtered_test.shape}")
print("\nRemoval summary:")
print(f"- NaN features removed: {len(nan_columns)}")
print(f"- Zero-variance features removed: {len(zero_variance_columns)}")
print(f"- High-correlation features removed: {len(correlation_drop_columns)}")


Saved filtered Train dataset: /home/jun/Documents/qsar_modeling/data/features/filtered_train.csv
Saved filtered Test dataset: /home/jun/Documents/qsar_modeling/data/features/filtered_test.csv
Final Train shape: (514, 381)
Final Test shape: (128, 381)

Removal summary:
- NaN features removed: 491
- Zero-variance features removed: 165
- High-correlation features removed: 578


## Step 3.2: 3-Track Clustering and Silhouette Scoring

This section applies three clustering tracks to the filtered Training-set descriptors only. It saves per-compound cluster labels for use in the 12-model PyQSAR3 execution matrix. No GA/MC feature selection or MLR/PLS modeling is performed here.

In [7]:
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

FILTERED_TRAIN = PROJECT_ROOT / "data" / "features" / "filtered_train.csv"
CLUSTER_LABELS_OUT = PROJECT_ROOT / "data" / "features" / "cluster_labels_train.csv"

N_CLUSTERS = 4
RANDOM_STATE = 42

cluster_train = pd.read_csv(FILTERED_TRAIN)
assert cluster_train.shape == (514, 381), f"Unexpected filtered train shape: {cluster_train.shape}"

smiles_train = cluster_train["SMILES"].copy()
y_train_cluster = cluster_train["logKoc"].copy()
X_train_cluster = cluster_train.drop(columns=["SMILES", "logKoc"]).copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_cluster)

print(f"Filtered Training shape: {cluster_train.shape}")
print(f"X_train clustering matrix: {X_train_scaled.shape}")
print(f"Number of clusters per method: {N_CLUSTERS}")


Filtered Training shape: (514, 381)
X_train clustering matrix: (514, 379)
Number of clusters per method: 4


### Track 1 and Track 2: Hierarchical and K-Means Clustering

Hierarchical clustering uses Ward linkage through `AgglomerativeClustering`. K-Means uses deterministic initialization through `random_state=42`.

In [8]:
hierarchical_model = AgglomerativeClustering(
    n_clusters=N_CLUSTERS,
    linkage="ward",
)
hierarchical_labels = hierarchical_model.fit_predict(X_train_scaled)

kmeans_model = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=20,
)
kmeans_labels = kmeans_model.fit_predict(X_train_scaled)

print("Hierarchical label counts:")
print(pd.Series(hierarchical_labels).value_counts().sort_index().to_string())
print("\nK-Means label counts:")
print(pd.Series(kmeans_labels).value_counts().sort_index().to_string())


Hierarchical label counts:
0    170
1    282
2     50
3     12

K-Means label counts:
0     48
1    165
2    183
3    118


### Track 3: SOM Clustering

If `minisom` is available, it is used directly. Otherwise, a deterministic NumPy SOM approximation with a 2x2 map is used to avoid changing the `pq3` environment.

In [9]:
def train_numpy_som(X: np.ndarray, grid_shape=(2, 2), n_iter=1200, random_state=RANDOM_STATE) -> np.ndarray:
    rng = np.random.default_rng(random_state)
    n_units = grid_shape[0] * grid_shape[1]
    weights = X[rng.choice(X.shape[0], size=n_units, replace=False)].copy()
    grid_positions = np.array([(i, j) for i in range(grid_shape[0]) for j in range(grid_shape[1])])

    for step in range(n_iter):
        sample = X[rng.integers(0, X.shape[0])]
        distances = np.linalg.norm(weights - sample, axis=1)
        bmu = int(np.argmin(distances))

        progress = step / max(n_iter - 1, 1)
        learning_rate = 0.5 * (1.0 - progress) + 0.05 * progress
        sigma = 1.0 * (1.0 - progress) + 0.2 * progress
        grid_distances = np.linalg.norm(grid_positions - grid_positions[bmu], axis=1)
        neighborhood = np.exp(-(grid_distances ** 2) / (2 * sigma ** 2))
        weights += learning_rate * neighborhood[:, None] * (sample - weights)

    final_distances = np.linalg.norm(X[:, None, :] - weights[None, :, :], axis=2)
    return np.argmin(final_distances, axis=1)


try:
    from minisom import MiniSom

    som = MiniSom(
        x=2,
        y=2,
        input_len=X_train_scaled.shape[1],
        sigma=1.0,
        learning_rate=0.5,
        random_seed=RANDOM_STATE,
    )
    som.random_weights_init(X_train_scaled)
    som.train_random(X_train_scaled, num_iteration=1200)
    som_labels = np.array([
        winner[0] * 2 + winner[1]
        for winner in (som.winner(row) for row in X_train_scaled)
    ])
    som_method = "MiniSom"
except Exception as exc:
    som_labels = train_numpy_som(X_train_scaled)
    som_method = f"NumPy SOM approximation ({exc})"

print(f"SOM implementation: {som_method}")
print("SOM label counts:")
print(pd.Series(som_labels).value_counts().sort_index().to_string())


SOM implementation: NumPy SOM approximation (No module named 'minisom')
SOM label counts:
0    238
1     71
2     84
3    121


### Silhouette Evaluation and Label Export

Silhouette scores are calculated on the scaled Training features for each clustering output. The resulting per-compound labels are saved for Step 3.3.

In [10]:
def checked_silhouette(X: np.ndarray, labels: np.ndarray) -> float:
    unique_labels = np.unique(labels)
    assert len(unique_labels) > 1, "Silhouette score requires at least two clusters"
    assert len(unique_labels) < len(labels), "Silhouette score requires fewer clusters than samples"
    return float(silhouette_score(X, labels))


silhouette_scores = {
    "Hierarchical": checked_silhouette(X_train_scaled, hierarchical_labels),
    "KMeans": checked_silhouette(X_train_scaled, kmeans_labels),
    "SOM": checked_silhouette(X_train_scaled, som_labels),
}

print("Silhouette Scores:")
for method, score in silhouette_scores.items():
    print(f"{method}: {score:.6f}")

cluster_labels_train = pd.DataFrame({
    "SMILES": smiles_train,
    "Hierarchical_Label": hierarchical_labels,
    "KMeans_Label": kmeans_labels,
    "SOM_Label": som_labels,
})

CLUSTER_LABELS_OUT.parent.mkdir(parents=True, exist_ok=True)
cluster_labels_train.to_csv(CLUSTER_LABELS_OUT, index=False)

print(f"Saved cluster labels: {CLUSTER_LABELS_OUT}")
print(f"Cluster label output shape: {cluster_labels_train.shape}")
display(cluster_labels_train.head())


Silhouette Scores:
Hierarchical: 0.128889
KMeans: 0.059917
SOM: 0.084810
Saved cluster labels: /home/jun/Documents/qsar_modeling/data/features/cluster_labels_train.csv
Cluster label output shape: (514, 4)


,SMILES,Hierarchical_Label,KMeans_Label,SOM_Label
0,C=CC=O,1,0,0
1,CC(=O)O,2,2,0
2,CNC(=O)/C=C(\C)OP(=O)(OC)OC,0,1,1
3,CCO,2,0,0
4,C[C@H](O)CO,2,0,0


## Step 3.2b: Multi-Track Data Partitioning

This intermediate preparation step physically separates the filtered Training data into three clustering-specific track files for the PyQSAR3 12-model matrix. Each output keeps `SMILES`, `logKoc`, all 379 filtered descriptors, and exactly one standardized `Cluster_Label` column.

In [11]:
FILTERED_TRAIN = PROJECT_ROOT / "data" / "features" / "filtered_train.csv"
CLUSTER_LABELS_TRAIN = PROJECT_ROOT / "data" / "features" / "cluster_labels_train.csv"

TRACK_OUTPUTS = {
    "Hierarchical_Label": PROJECT_ROOT / "data" / "features" / "train_track_hierarchical.csv",
    "KMeans_Label": PROJECT_ROOT / "data" / "features" / "train_track_kmeans.csv",
    "SOM_Label": PROJECT_ROOT / "data" / "features" / "train_track_som.csv",
}

filtered_train_for_tracks = pd.read_csv(FILTERED_TRAIN)
cluster_labels_for_tracks = pd.read_csv(CLUSTER_LABELS_TRAIN)

assert filtered_train_for_tracks.shape == (514, 381), f"Unexpected filtered_train shape: {filtered_train_for_tracks.shape}"
assert cluster_labels_for_tracks.shape == (514, 4), f"Unexpected cluster_labels shape: {cluster_labels_for_tracks.shape}"
assert filtered_train_for_tracks["SMILES"].is_unique, "Filtered train SMILES are not unique"
assert cluster_labels_for_tracks["SMILES"].is_unique, "Cluster label SMILES are not unique"

merged_tracks = filtered_train_for_tracks.merge(
    cluster_labels_for_tracks,
    on="SMILES",
    how="inner",
    validate="one_to_one",
)

expected_merged_columns = filtered_train_for_tracks.shape[1] + 3
assert merged_tracks.shape == (514, expected_merged_columns), f"Unexpected merged shape: {merged_tracks.shape}"

track_shapes = {}
for label_column, output_path in TRACK_OUTPUTS.items():
    track_df = filtered_train_for_tracks.merge(
        cluster_labels_for_tracks[["SMILES", label_column]],
        on="SMILES",
        how="inner",
        validate="one_to_one",
    )
    track_df = track_df.rename(columns={label_column: "Cluster_Label"})

    assert track_df.shape == (514, 382), f"Unexpected {label_column} track shape: {track_df.shape}"
    assert track_df.columns[-1] == "Cluster_Label", "Cluster_Label must be the final column"
    assert track_df["Cluster_Label"].notna().all(), f"Missing labels in {label_column}"
    assert track_df.drop(columns=["Cluster_Label"]).equals(filtered_train_for_tracks), (
        f"Feature data changed while creating {label_column}"
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    track_df.to_csv(output_path, index=False)
    track_shapes[output_path.name] = track_df.shape

print("Saved multi-track Training datasets:")
for file_name, shape in track_shapes.items():
    print(f"- {file_name}: {shape}")


Saved multi-track Training datasets:
- train_track_hierarchical.csv: (514, 382)
- train_track_kmeans.csv: (514, 382)
- train_track_som.csv: (514, 382)
